# Tarea 2 — Estimar y predecir olas estacionales con un modelo SIRS

En la clase 2 ajustamos un modelo con una tasa de transmisión constante, $\beta$. En esta tarea usaremos datos semanales de tres temporadas y supondremos que la transmisión cambia periódicamente durante el año.

## Objetivos

Al terminar deberás poder:

1. implementar una tasa de transmisión estacional $\beta(t;\theta)$;
2. estimar $\theta=(\theta_0,\theta_s,\theta_c)$ con las primeras dos temporadas;
3. usar el modelo ajustado para predecir la tercera temporada;
4. evaluar la predicción con el RMSE.

Completa las celdas marcadas con **TODO**. Trabaja en orden y conserva tus respuestas breves en las celdas de Markdown.

In [ ]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 12,
})

## 1. Modelo SIRS con transmisión estacional

Un modelo SIR cerrado no suele producir olas recurrentes grandes: después de un brote quedan pocos susceptibles. Para representar varias temporadas, incorporamos pérdida de inmunidad, de modo que las personas recuperadas regresan al compartimento susceptible:

$$
\frac{dS}{dt}=-\beta(t)\frac{SI}{N}+\omega R,
\qquad
\frac{dI}{dt}=\beta(t)\frac{SI}{N}-\gamma I,
\qquad
\frac{dR}{dt}=\gamma I-\omega R.
$$

La tasa de transmisión tiene periodo conocido $T=52$ semanas:

$$
\beta(t;\theta)=
\exp\left[
\theta_0+
\theta_s\sin\left(\frac{2\pi t}{T}\right)+
\theta_c\cos\left(\frac{2\pi t}{T}\right)
\right].
$$

La exponencial garantiza que $\beta(t;\theta)>0$. En esta tarea $N$, $\gamma$, $\omega$, $T$ y la condición inicial son conocidos; sólo estimaremos $\theta=(\theta_0,\theta_s,\theta_c)$.

### Incidencia semanal

En SIRS no podemos calcular la incidencia como $S(t_{j-1})-S(t_j)$ porque $S(t)$ también puede aumentar debido al flujo $R\to S$. Usaremos un acumulador de infecciones $C(t)$:

$$
\frac{dC}{dt}=\beta(t)\frac{S(t)I(t)}{N},
\qquad
G_j(\theta)=C(t_j)-C(t_{j-1}).
$$

**Pregunta 1.** Explica en dos o tres líneas por qué $-\mathrm{diff}(S)$ no representa necesariamente las infecciones nuevas en un modelo SIRS.

> **Respuesta:** Escribe aquí tu respuesta.

In [ ]:
# La unidad de tiempo es una semana.
N = 100_000
T = 52.0
gamma = 1 / 2       # periodo infeccioso medio: 2 semanas
omega = 1 / 52      # duración media de la inmunidad: 1 año

n_total = 3 * int(T)
n_entrenamiento = 2 * int(T)

# La enfermedad ya circulaba antes de t = 0.
I0 = 100.0
R0 = 49_900.0
S0 = N - I0 - R0
y0 = np.array([S0, I0, R0, 0.0])

## 2. Ejercicio 1 — Implementa la transmisión estacional

Completa `beta_estacional`. La entrada `t` puede ser un número o un arreglo de semanas, y `theta` contiene los tres parámetros $(\theta_0,\theta_s,\theta_c)$.

**Pregunta 2.** ¿Por qué conviene usar la exponencial en la definición de $\beta(t;\theta)$?

> **Respuesta:** Escribe aquí tu respuesta.

In [ ]:
def beta_estacional(t, theta):
    """Tasa de transmisión semanal, siempre positiva."""
    theta_0, theta_s, theta_c = theta

    # TODO 1: define la fase 2*pi*t/T.
    # TODO 2: devuelve beta(t; theta) usando la fórmula de la sección anterior.

    raise NotImplementedError('Completa beta_estacional antes de continuar.')

### Funciones dadas para resolver el modelo

Estas funciones implementan el sistema SIRS y el operador directo $G(\theta)$. Léelas con atención: observa en particular que la incidencia se calcula con `np.diff(C)`, donde $C$ es el acumulador de infecciones.

In [ ]:
def sirs_estacional(t, y, theta):
    """Lado derecho del modelo SIRS con acumulador de incidencia."""
    S, I, R, C = y
    beta_t = beta_estacional(t, theta)
    infecciones_nuevas = beta_t * S * I / N

    dS = -infecciones_nuevas + omega * R
    dI = infecciones_nuevas - gamma * I
    dR = gamma * I - omega * R
    dC = infecciones_nuevas
    return [dS, dI, dR, dC]


def resolver_sirs(theta, n_semanas, y_inicial=y0):
    """Resuelve el modelo y devuelve estados e incidencia semanal."""
    t_eval = np.arange(n_semanas + 1, dtype=float)
    solucion = solve_ivp(
        sirs_estacional,
        t_span=(0, n_semanas),
        y0=y_inicial,
        args=(theta,),
        t_eval=t_eval,
        method='LSODA',
        rtol=1e-8,
        atol=1e-6,
    )
    if not solucion.success:
        raise RuntimeError(solucion.message)

    S, I, R, C = solucion.y
    return {
        'semana': t_eval,
        'S': S,
        'I': I,
        'R': R,
        'C': C,
        'semana_incidencia': t_eval[1:],
        'incidencia': np.diff(C),
    }


def G(theta, n_semanas):
    """Operador directo: incidencia semanal predicha."""
    return resolver_sirs(theta, n_semanas)['incidencia']


def resumen_estacional(theta):
    """Resume nivel, amplitud y semana del máximo de beta(t)."""
    theta_0, theta_s, theta_c = theta
    amplitud_log = np.hypot(theta_s, theta_c)
    fase = np.arctan2(theta_c, theta_s)
    semana_maxima = ((np.pi / 2 - fase) * T / (2 * np.pi)) % T
    return {
        'nivel_basal_geometrico': np.exp(theta_0),
        'amplitud_log': amplitud_log,
        'beta_minima': np.exp(theta_0 - amplitud_log),
        'beta_maxima': np.exp(theta_0 + amplitud_log),
        'semana_maxima': semana_maxima,
    }

## 3. Cargar y explorar los datos

El archivo contiene casos semanales observados durante tres temporadas. No conocemos la trayectoria determinista que generó estos datos.

In [ ]:
datos = pd.read_csv('datos_tarea_02_sirs_estacional.csv')
semanas_incidencia = datos['semana'].to_numpy()
casos_observados = datos['casos_observados'].to_numpy()

display(datos.head())
print(f'Número de semanas observadas: {len(datos)}')

In [ ]:
fig, ax = plt.subplots()
ax.scatter(semanas_incidencia, casos_observados, color='#E05C47', s=18, alpha=0.75,
           label='Casos observados', zorder=3)
for corte in (52, 104):
    ax.axvline(corte, color='0.45', ls=':', lw=1)
ax.set(xlabel='Semana', ylabel='Casos nuevos',
       title='Casos observados durante tres temporadas')
ax.legend(frameon=False, loc='best')
plt.tight_layout()
plt.show()

## 4. Entrenamiento y prueba

Usaremos las primeras dos temporadas, semanas $1,\ldots,104$, para ajustar $\theta$. La tercera temporada, semanas $105,\ldots,156$, queda retenida para evaluar si el patrón estimado anticipa la siguiente ola.

Al predecir la tercera temporada se conserva el mismo reloj: no se reinicia $t$ en la semana 105, porque la fase estacional depende de $t\bmod T$.

In [ ]:
mask_train = np.arange(n_total) < n_entrenamiento

fig, ax = plt.subplots()
ax.axvspan(0, n_entrenamiento, color='#D9EAF7', alpha=0.7, label='Entrenamiento')
ax.axvspan(n_entrenamiento, n_total, color='#F2F2F2', alpha=0.9, label='Prueba')
ax.scatter(semanas_incidencia[mask_train], casos_observados[mask_train],
           color='#E05C47', s=20, label='Casos de entrenamiento', zorder=3)
ax.scatter(semanas_incidencia[~mask_train], casos_observados[~mask_train],
           color='0.35', s=20, label='Casos de prueba', zorder=3)
ax.axvline(n_entrenamiento, color='0.25', ls='--', zorder=2)
ax.set(xlabel='Semana', ylabel='Casos nuevos',
       title='Dos temporadas para ajustar y una para predecir')
ax.legend(frameon=False, loc='best', ncol=2)
plt.tight_layout()
plt.show()

## 5. Ejercicio 2 — Ajuste por mínimos cuadrados

Para cada vector $\theta$, el operador directo $G(\theta)$ devuelve la incidencia semanal predicha. Estimaremos los parámetros minimizando la discrepancia sólo en el periodo de entrenamiento:

$$
Q_{\mathrm{ent}}(\theta)
=
\sum_{j=1}^{104}\left[Y_j-G_j(\theta)\right]^2,
\qquad
\widehat\theta=
\underset{\theta}{\operatorname*{argmin}}\;Q_{\mathrm{ent}}(\theta).
$$

Completa la función `perdida`. Las cotas, el punto inicial y el optimizador se proporcionan para concentrarnos en la construcción de la función objetivo.

In [ ]:
def perdida(theta, observaciones, n_entrenamiento):
    # TODO 1: calcula la incidencia predicha para las semanas de entrenamiento.
    # TODO 2: calcula los residuos entre observaciones y predicción.
    # TODO 3: devuelve la suma de los cuadrados de los residuos.

    raise NotImplementedError('Completa la función de pérdida antes de optimizar.')


cotas = [
    (np.log(0.7), np.log(1.6)),
    (-0.75, 0.75),
    (-0.75, 0.75),
]

# Cuando hayas completado `perdida`, ejecuta esta celda sin modificar
# los argumentos del optimizador.
ajuste = minimize(
    fun=perdida,
    x0=np.zeros(3),
    args=(casos_observados, n_entrenamiento),
    method='L-BFGS-B',
    bounds=cotas,
    options={'maxiter': 500},
)

if not ajuste.success:
    raise RuntimeError(ajuste.message)

theta_estimado = ajuste.x
modelo_ajustado = resolver_sirs(theta_estimado, n_total)
incidencia_ajustada = modelo_ajustado['incidencia']

print(f'Optimización terminada: {ajuste.message}')
print(f'Valor de la función objetivo: {ajuste.fun:,.0f}')

## 6. ¿Qué patrón de transmisión estima el modelo?

La siguiente celda muestra $\widehat\beta(t)$. Reporta la semana anual en la que la transmisión estimada es máxima y el rango estimado de $\beta(t)$.

**Pregunta 3.** ¿En qué semana del año es máxima la transmisión estimada? ¿Qué significa esa semana en términos del modelo?

In [ ]:
tabla_parametros = pd.DataFrame({
    'parámetro': [r'$\theta_0$', r'$\theta_s$', r'$\theta_c$'],
    'estimado': theta_estimado,
})
display(tabla_parametros.style.format({'estimado': '{:.4f}'}))

resumen_estimado = resumen_estacional(theta_estimado)
print(f"Semana estimada de máximo de beta: {resumen_estimado['semana_maxima']:.1f}")
print(f"Rango estimado de beta: ({resumen_estimado['beta_minima']:.3f}, "
      f"{resumen_estimado['beta_maxima']:.3f}) por semana")

t_modelo = np.arange(n_total + 1)
fig, ax = plt.subplots()
ax.axvspan(0, n_entrenamiento, color='#D9EAF7', alpha=0.55, label='Entrenamiento')
ax.axvspan(n_entrenamiento, n_total, color='#F2F2F2', alpha=0.8, label='Prueba')
ax.plot(t_modelo, beta_estacional(t_modelo, theta_estimado),
        color='#2878B5', lw=2.8, label=r'$\widehat{\beta}(t)$')
ax.axvline(n_entrenamiento, color='0.25', ls='--')
ax.set(xlabel='Semana', ylabel=r'$\beta(t)$ por semana',
       title='Patrón estacional de transmisión estimado')
ax.legend(frameon=False, loc='best')
plt.tight_layout()
plt.show()

## 7. Predicción de la tercera temporada

La curva azul que construiremos a continuación se ajusta sólo con las primeras 104 semanas. Al continuar la simulación hasta la semana 156, el modelo pronostica la tercera ola sin usar sus observaciones para estimar $\theta$.

In [ ]:
fig, ax = plt.subplots()
ax.axvspan(0, n_entrenamiento, color='#D9EAF7', alpha=0.7, label='Entrenamiento')
ax.axvspan(n_entrenamiento, n_total, color='#F2F2F2', alpha=0.9, label='Prueba')
ax.scatter(semanas_incidencia[mask_train], casos_observados[mask_train],
           color='#E05C47', s=20, label='Casos de entrenamiento', zorder=4)
ax.scatter(semanas_incidencia[~mask_train], casos_observados[~mask_train],
           color='0.35', s=18, label='Casos de prueba', zorder=4)
ax.plot(semanas_incidencia, incidencia_ajustada, color='#2878B5', lw=2.8,
        label='Predicción ajustada', zorder=3)
ax.axvline(n_entrenamiento, color='0.25', ls='--', label='Fin del entrenamiento')
ax.set(xlabel='Semana', ylabel='Casos nuevos',
       title='Ajuste con dos temporadas y predicción de la tercera')
ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.16),
          ncol=3, fontsize=10)
fig.subplots_adjust(bottom=0.27)
plt.show()

## 8. Ejercicio 3 — Evalúa la predicción

Completa los dos RMSE:

$$
\operatorname{RMSE}=
\sqrt{\frac{1}{m}\sum_{j=1}^{m}(Y_j-G_j)^2}.
$$

**Pregunta 4.** Compara el RMSE de entrenamiento con el de prueba. ¿Qué sugiere esa comparación sobre la capacidad de generalización del modelo?

In [ ]:
# TODO 1: calcula el RMSE en entrenamiento.
# TODO 2: calcula el RMSE en prueba.

raise NotImplementedError('Completa los dos RMSE.')

## Responde brevemente:

1. ¿Por qué no podemos usar $S(t_{j-1})-S(t_j)$ como incidencia en SIRS?
2. ¿Por qué la exponencial en $\beta(t;\theta)$ es importante?
3. Reporta $\widehat\theta$ y la semana anual de máxima transmisión estimada.
4. Incluye la gráfica de ajuste y predicción de la tercera temporada, junto con ambos RMSE.
5. ¿Qué supuesto permite usar las primeras dos temporadas para pronosticar la tercera?
6. Menciona una fuente de incertidumbre que este ajuste no incorpora.

### Extensión opcional

Compara el modelo estacional contra uno sin estacionalidad, imponiendo $\theta_s=\theta_c=0$. ¿Cómo cambia el RMSE de prueba?